# Practical No. 06 — Retrieval-Augmented Generation (RAG) using a Pre-trained LLM

**Project:** AI & ML Academic Knowledge Assistant  
**Platform:** Google Colab  
**Core stack:** Python · PyPDF · Sentence Transformers · FAISS · FLAN-T5 · Gradio

| Student detail | Fill before submission |
|---|---|
| Student name | PARTH KITCHLOO |
| PRN / Roll number | 202401110014 |
| Branch / Batch | AIML,A2 |
| Subject / Practical | Practical No. 06 — RAG using Pre-trained LLM |

## Aim
To build a document-grounded question-answering application that loads a domain-specific document collection, preprocesses and chunks the text, creates semantic embeddings, indexes them in a vector database, retrieves relevant passages, and uses a pretrained Large Language Model (LLM) to generate answers. The RAG output is compared with the same LLM operating without retrieval.

## Objectives
1. Load domain-specific PDF documents and extract their text.
2. Clean the extracted text and divide it into overlapping chunks.
3. Generate dense vector embeddings using a pretrained sentence-embedding model.
4. Build a FAISS vector index and retrieve the most relevant chunks for a question.
5. Integrate a pretrained FLAN-T5 model to generate answers from retrieved context.
6. Generate a baseline answer from the same LLM without retrieved context.
7. Compare both approaches using relevance, correctness, grounding, completeness, and hallucination control.
8. Demonstrate the pipeline through an interactive Gradio interface.

## Problem statement
A general-purpose LLM may not know the contents of a student's specific notes or may produce unsupported details. This project addresses that problem by retrieving relevant passages from a selected knowledge base and supplying them to a pretrained LLM as context. The system also keeps an LLM-only baseline so that the effect and limitations of retrieval can be examined.

## RAG workflow

```text
PDF knowledge base → text extraction → cleaning → chunking
        → embedding model → FAISS vector index
User question → question embedding → similarity search
        → top-k retrieved passages + question → pretrained LLM
        → grounded answer + source references

Baseline: User question → same pretrained LLM (no retrieval)
```

## Important notes before running
- Run the notebook from top to bottom in **Google Colab**. A GPU runtime is recommended but not compulsory: **Runtime → Change runtime type → T4 GPU** (when available).
- The notebook includes a small built-in AI/ML starter corpus so it can be demonstrated even before you upload files. For a stronger submission, upload **5–10 relevant, text-based AI/ML PDFs** in the PDF upload cell; the uploaded PDFs are added to the starter corpus.
- Model weights are downloaded from Hugging Face the first time. An internet connection is required. Download time depends on Colab and network availability.
- No model training is performed. The embedding model and FLAN-T5 are pretrained, as required by the practical.
- Generation is deliberately bounded (`max_new_tokens`, fixed beam count, and input truncation); the notebook does not contain an unbounded generation loop.
- Scanned/image-only PDFs may not contain extractable text. This notebook reports pages with no text; use searchable PDFs or OCR them separately before uploading.


## 1. Install required libraries

This cell installs only the libraries used in the practical. It may take a few minutes on the first run. If Colab displays a message asking to restart the session after installation, restart once and then continue from the imports cell.

In [1]:
%pip -q install "pypdf>=5.0" "sentence-transformers>=3.0" "faiss-cpu>=1.8" "transformers>=4.45" "sentencepiece>=0.2" "gradio>=5.0" "pandas>=2.0"

## 2. Import libraries and configure the runtime

We use `torch.cuda.is_available()` to select a GPU when Colab provides one. The same code also supports CPU execution.

In [2]:
import os
import re
import time
import warnings
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files
from IPython.display import display

# Keep tokenizer output tidy and bound Hugging Face network timeouts.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python: {os.sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"FAISS: {faiss.__version__ if hasattr(faiss, '__version__') else 'installed'}")
print(f"Compute device: {DEVICE}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available — CPU mode")

Python: 3.13.16
PyTorch: 2.11.0+cu130
FAISS: 1.15.1
Compute device: cuda
GPU: Tesla T4


## 3. Prepare the domain-specific document collection

The project domain is **Artificial Intelligence and Machine Learning**. The following short starter notes make the notebook reproducible. Uploading your course PDFs is strongly recommended: the uploaded pages are included alongside these starter notes, and retrieval metadata keeps the source filename and page number.

**PDF input limits for predictable processing:** the code accepts at most 10 PDFs, skips files larger than 25 MB, and processes at most 150 pages per PDF. Upload searchable PDFs rather than scanned image-only files.

In [3]:
STARTER_DOCS = [
    {
        "source": "Starter Notes - AI and Machine Learning Fundamentals",
        "page": 1,
        "text": """Artificial intelligence (AI) is the broad field of building computer systems that perform tasks associated with intelligent behaviour, such as perception, reasoning, language understanding, and decision support. Machine learning (ML) is a subfield of AI in which systems learn patterns from data rather than relying only on hand-written rules. A machine-learning workflow commonly includes defining a problem, collecting data, cleaning and exploring data, selecting features, splitting data into training and test sets, selecting an algorithm, training, evaluating, and monitoring the model. A model should be evaluated on data that was not used to fit it so that its ability to generalise can be estimated.

Deep learning is a branch of machine learning that uses neural networks with multiple layers. Natural language processing (NLP) focuses on computational methods for working with human language. These fields overlap: for example, a deep neural network can be used for text classification, image recognition, or speech processing. The choice of method depends on the task, the available data, interpretability needs, computational resources, and the cost of incorrect predictions."""
    },
    {
        "source": "Starter Notes - Supervised Learning",
        "page": 1,
        "text": """Supervised learning uses examples containing input features and known target labels. The model learns a mapping from inputs to targets and is evaluated on unseen examples. Classification predicts a discrete class, such as spam or not spam. Regression predicts a continuous numerical value, such as a house price. Common supervised algorithms include linear regression, logistic regression, decision trees, random forests, support vector machines, k-nearest neighbours, and neural networks.

A dataset is commonly divided into training, validation, and test sets. The training set is used to fit model parameters. The validation set helps choose hyperparameters and compare model versions. The test set is held aside until final evaluation. Data leakage occurs when information unavailable at prediction time, or information from the test set, influences training. Leakage can make evaluation scores misleadingly high. Preprocessing steps should be fitted on training data and then applied to validation and test data."""
    },
    {
        "source": "Starter Notes - Unsupervised Learning",
        "page": 1,
        "text": """Unsupervised learning works with data that does not have labelled target outputs. The aim is to discover structure or useful representations in the data. Clustering groups similar observations. K-means is a widely used clustering method that assigns observations to a chosen number of clusters by iteratively updating cluster centroids. The number of clusters, feature scaling, outliers, and initial centroid placement can affect its result. Hierarchical clustering builds a hierarchy of nested clusters. Dimensionality reduction represents data using fewer variables while attempting to preserve important structure; principal component analysis (PCA) is a common linear method.

Clustering results require interpretation because cluster labels do not automatically correspond to meaningful real-world categories. Unsupervised learning is used for customer segmentation, grouping documents, exploratory data analysis, anomaly exploration, and representation learning."""
    },
    {
        "source": "Starter Notes - Model Evaluation and Overfitting",
        "page": 1,
        "text": """A confusion matrix compares predicted classes with actual classes. For binary classification, true positives (TP) are positive cases correctly predicted as positive; true negatives (TN) are negative cases correctly predicted as negative; false positives (FP) are negative cases incorrectly predicted as positive; and false negatives (FN) are positive cases incorrectly predicted as negative. Accuracy is (TP + TN) / (TP + TN + FP + FN). Precision is TP / (TP + FP), and recall is TP / (TP + FN). The F1-score is the harmonic mean of precision and recall: 2 × precision × recall / (precision + recall). The metric should match the task and the cost of different errors; accuracy alone can be misleading for imbalanced datasets.

Overfitting occurs when a model fits training examples too closely, including noise, and performs poorly on unseen data. Underfitting occurs when a model is too simple to capture important patterns. Generalisation can be improved through an appropriate train-validation-test split, cross-validation, regularisation, early stopping, feature selection, data augmentation where appropriate, and collecting representative data. A large gap between training and test performance can be a warning sign of overfitting."""
    },
    {
        "source": "Starter Notes - Neural Networks and Deep Learning",
        "page": 1,
        "text": """An artificial neural network is made of layers of connected computational units often called neurons. A typical feed-forward network contains an input layer, one or more hidden layers, and an output layer. Each neuron computes a weighted sum of its inputs plus a bias and then applies an activation function. Common activation functions include ReLU for hidden layers and sigmoid or softmax for some output tasks. During training, a loss function measures prediction error. Backpropagation computes gradients of the loss with respect to model parameters, and an optimiser such as stochastic gradient descent or Adam updates those parameters.

A learning rate controls the size of parameter updates. If it is too high, training may become unstable; if it is too low, learning may be slow. Deep learning can model complex patterns but often requires substantial data and compute. Dropout, weight decay, early stopping, and suitable data splits are common ways to reduce overfitting. Results should be compared against simpler baselines where appropriate."""
    },
    {
        "source": "Starter Notes - Embeddings, Retrieval and RAG",
        "page": 1,
        "text": """An embedding is a dense numerical vector that represents information such as a sentence or paragraph. Texts with related meanings can have nearby vectors in an embedding space. A sentence-transformer model encodes passages and questions into vectors. To retrieve relevant passages, the system embeds the user's question and compares it with stored passage vectors. Cosine similarity measures the angle-based similarity between two non-zero vectors. When vectors are normalised to unit length, their dot product equals cosine similarity.

Retrieval-Augmented Generation (RAG) combines a retrieval step with a text-generation step. Documents are extracted, cleaned, split into chunks, embedded, and indexed in a vector store. At question time, the system retrieves the top-k most similar chunks and adds them to a prompt for a pretrained language model. The model generates an answer using the supplied context. The retrieved sources can be shown to users for traceability. RAG can make answers more specific to a document collection, but it does not guarantee correctness: retrieval can miss relevant passages, chunks can omit context, source documents can contain errors, and the language model can still produce unsupported statements. An LLM-only baseline answers without searching the collection and is useful for comparison."""
    }
]

MAX_PDFS = 10
MAX_FILE_MB = 25
MAX_PAGES_PER_PDF = 150

print(f"Starter documents ready: {len(STARTER_DOCS)}")
print("Next, upload your searchable AI/ML PDF notes or textbooks in the following cell.")

Starter documents ready: 6
Next, upload your searchable AI/ML PDF notes or textbooks in the following cell.


## 4. Upload PDFs and extract their text

When the upload dialog opens, select up to 10 relevant PDFs. If you do not upload a file, the notebook continues with the built-in AI/ML starter notes. For each PDF, the extractor records the source filename and page number so retrieved passages can be traced.

In [5]:
uploaded_files = files.upload()  # Select PDF files; selecting none uses the starter notes.

raw_documents = list(STARTER_DOCS)  # Keep the reproducible AI/ML starter knowledge base.
extraction_warnings = []
accepted_pdf_count = 0

for filename, file_bytes in list(uploaded_files.items())[:MAX_PDFS]:
    if not filename.lower().endswith(".pdf"):
        extraction_warnings.append(f"Skipped {filename}: file is not a PDF.")
        continue
    if len(file_bytes) > MAX_FILE_MB * 1024 * 1024:
        extraction_warnings.append(f"Skipped {filename}: larger than {MAX_FILE_MB} MB.")
        continue
    try:
        reader = PdfReader(BytesIO(file_bytes))
        if reader.is_encrypted:
            try:
                reader.decrypt("")
            except Exception:
                extraction_warnings.append(f"Skipped {filename}: encrypted PDF could not be opened.")
                continue
        page_count = min(len(reader.pages), MAX_PAGES_PER_PDF)
        if len(reader.pages) > MAX_PAGES_PER_PDF:
            extraction_warnings.append(
                f"{filename}: processed first {MAX_PAGES_PER_PDF} of {len(reader.pages)} pages."
            )
        pages_with_text = 0
        for page_idx in range(page_count):
            page_text = reader.pages[page_idx].extract_text() or ""
            page_text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", page_text)
            page_text = re.sub(r"\s+", " ", page_text).strip()
            if page_text:
                raw_documents.append({
                    "source": filename,
                    "page": page_idx + 1,
                    "text": page_text
                })
                pages_with_text += 1
        accepted_pdf_count += 1
        if pages_with_text == 0:
            extraction_warnings.append(
                f"{filename}: no selectable text found. It may be a scanned PDF; use a searchable/OCR-processed PDF."
            )
    except Exception as exc:
        extraction_warnings.append(f"Could not read {filename}: {type(exc).__name__}: {exc}")

print(f"PDF files accepted for processing: {accepted_pdf_count}")
print(f"Document pages/sections available: {len(raw_documents)}")
for warning in extraction_warnings:
    print("WARNING:", warning)

if not raw_documents:
    raise RuntimeError("No documents are available. Upload a readable PDF or restore STARTER_DOCS.")

print("\nSample source records:")
display(pd.DataFrame(raw_documents)[["source", "page", "text"]].head(5))

Saving 1.pdf to 1.pdf
Saving 2.pdf to 2.pdf
Saving 3.pdf to 3.pdf
PDF files accepted for processing: 2
Document pages/sections available: 58

Sample source records:


,source,page,text
0,Starter Notes - AI and Machine Learning Fundam...,1,Artificial intelligence (AI) is the broad fiel...
1,Starter Notes - Supervised Learning,1,Supervised learning uses examples containing i...
2,Starter Notes - Unsupervised Learning,1,Unsupervised learning works with data that doe...
3,Starter Notes - Model Evaluation and Overfitting,1,A confusion matrix compares predicted classes ...
4,Starter Notes - Neural Networks and Deep Learning,1,An artificial neural network is made of layers...


## 5. Clean the text and create overlapping chunks

Large documents are split into smaller passages because embedding models and LLM prompts have input-length limits. This notebook uses a transparent word-based splitter:

- **Chunk size:** 180 words
- **Chunk overlap:** 40 words

Overlap preserves some information between adjacent chunks. The values are sensible starting settings rather than universal optimums; they can be adjusted based on document structure and retrieval quality.

In [6]:
CHUNK_SIZE_WORDS = 180
CHUNK_OVERLAP_WORDS = 40


def clean_text(text):
    """Normalise whitespace and remove common extraction artefacts."""
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"(?<=\w)-\s+(?=\w)", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_into_chunks(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    """Split text into bounded overlapping word chunks."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")
    words = clean_text(text).split()
    if not words:
        return []
    chunks_out = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        if not piece:
            break
        chunks_out.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break
    return chunks_out


chunks = []
for doc in raw_documents:
    cleaned = clean_text(doc.get("text", ""))
    for part_no, chunk_text in enumerate(split_into_chunks(cleaned), start=1):
        chunks.append({
            "text": chunk_text,
            "source": doc.get("source", "Unknown source"),
            "page": doc.get("page", "N/A"),
            "chunk_in_page": part_no
        })

if not chunks:
    raise RuntimeError("Text extraction produced no usable chunks. Check the uploaded PDFs.")

print(f"Total chunks created: {len(chunks)}")
print(f"Chunk size: {CHUNK_SIZE_WORDS} words | Overlap: {CHUNK_OVERLAP_WORDS} words")
display(pd.DataFrame(chunks)[["source", "page", "chunk_in_page", "text"]].head(8))

Total chunks created: 135
Chunk size: 180 words | Overlap: 40 words


,source,page,chunk_in_page,text
0,Starter Notes - AI and Machine Learning Fundam...,1,1,Artificial intelligence (AI) is the broad fiel...
1,Starter Notes - Supervised Learning,1,1,Supervised learning uses examples containing i...
2,Starter Notes - Unsupervised Learning,1,1,Unsupervised learning works with data that doe...
3,Starter Notes - Model Evaluation and Overfitting,1,1,A confusion matrix compares predicted classes ...
4,Starter Notes - Model Evaluation and Overfitting,1,2,when a model is too simple to capture importan...
5,Starter Notes - Neural Networks and Deep Learning,1,1,An artificial neural network is made of layers...
6,"Starter Notes - Embeddings, Retrieval and RAG",1,1,An embedding is a dense numerical vector that ...
7,"Starter Notes - Embeddings, Retrieval and RAG",1,2,users for traceability. RAG can make answers m...


## 6. Generate semantic embeddings

We use the pretrained `sentence-transformers/all-MiniLM-L6-v2` model. It maps each text chunk to a 384-dimensional dense vector. The same model must be used for both document chunks and user questions so that their vectors share the same embedding space.

Vectors are L2-normalised. This allows an inner-product FAISS index to rank them by cosine similarity.

In [7]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

chunk_texts = [item["text"] for item in chunks]
embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

if embeddings.ndim != 2 or embeddings.shape[0] != len(chunks):
    raise RuntimeError("Embedding generation returned an unexpected shape.")

print("Embedding matrix shape:", embeddings.shape)
print("Embedding dimensions:", embeddings.shape[1])
print("All embeddings finite:", bool(np.isfinite(embeddings).all()))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding matrix shape: (135, 384)
Embedding dimensions: 384
All embeddings finite: True


## 7. Build the FAISS vector database

FAISS (`IndexFlatIP`) stores the normalised embeddings and performs exact inner-product search. Since the vectors are normalised, the returned inner-product scores are cosine-similarity scores. Exact search is suitable for this practical's small or medium document collection and does not require training a separate index.

In [8]:
vector_dimension = embeddings.shape[1]
vector_index = faiss.IndexFlatIP(vector_dimension)
vector_index.add(embeddings)

assert vector_index.ntotal == len(chunks), "Vector count must match chunk count."
print(f"FAISS index created successfully. Indexed vectors: {vector_index.ntotal}")
print("Index type: IndexFlatIP (exact inner-product search over normalised vectors)")

FAISS index created successfully. Indexed vectors: 135
Index type: IndexFlatIP (exact inner-product search over normalised vectors)


## 8. Implement semantic document retrieval

For every user question, the retriever embeds the question, searches FAISS, and returns the top-*k* chunks. Each result includes its source, page, chunk identifier, and similarity score. The default is **top-k = 4**; it is capped automatically when the knowledge base contains fewer chunks.

In [9]:
DEFAULT_TOP_K = 4


def retrieve_documents(question, top_k=DEFAULT_TOP_K):
    """Retrieve the most similar chunks for a non-empty question."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Please enter a non-empty question.")
    if vector_index.ntotal == 0:
        return []
    k = max(1, min(int(top_k), vector_index.ntotal))
    query_vector = embedding_model.encode(
        [question.strip()],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)
    scores, positions = vector_index.search(query_vector, k)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos < 0:
            continue
        item = dict(chunks[int(pos)])
        item["similarity"] = float(score)
        item["chunk_id"] = int(pos)
        results.append(item)
    return results


# Retrieval smoke test (does not call the LLM).
retrieval_test_question = "What is the difference between classification and regression?"
retrieval_test_results = retrieve_documents(retrieval_test_question, top_k=4)
print("Question:", retrieval_test_question)
display(pd.DataFrame(retrieval_test_results)[["source", "page", "similarity", "text"]])

Question: What is the difference between classification and regression?


,source,page,similarity,text
0,Starter Notes - Supervised Learning,1,0.461340,Supervised learning uses examples containing i...
1,2.pdf,2,0.368796,What is Machine Learning (ML)? There are many ...
2,Starter Notes - AI and Machine Learning Fundam...,1,0.301826,Artificial intelligence (AI) is the broad fiel...
3,3.pdf,32,0.294457,(both potentially beneficial and harmful) base...


## 9. Load a pretrained Large Language Model (FLAN-T5)

The generator is `google/flan-t5-base`, a pretrained sequence-to-sequence instruction-tuned text model. It is not trained from scratch in this practical. A GPU is used when available; otherwise the model runs on CPU. The model-loading cell may take time on its first run because the weights and tokenizer are downloaded once and cached in the Colab session.

In [10]:
LLM_MODEL_NAME = "google/flan-t5-base"
LLM_READY = False
llm_tokenizer = None
llm_model = None
LLM_DEVICE = DEVICE

try:
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
    llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_NAME)
    llm_model = llm_model.to(LLM_DEVICE)
    llm_model.eval()
    LLM_READY = True
    print(f"Loaded pretrained model: {LLM_MODEL_NAME}")
    print(f"LLM device: {LLM_DEVICE}")
except Exception as exc:
    llm_tokenizer = None
    llm_model = None
    LLM_READY = False
    print("The pretrained LLM could not be loaded in this session.")
    print(f"Reason: {type(exc).__name__}: {exc}")
    print("Check Colab internet access, then rerun this cell. No generated answers will be fabricated.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded pretrained model: google/flan-t5-base
LLM device: cuda


## 10. Create bounded answer-generation functions

Two prompts are used with the **same FLAN-T5 model**:

- **RAG prompt:** includes retrieved passages and instructs the model to stay within that context. It asks the model to state when the documents do not contain the answer.
- **LLM-only prompt:** contains the question but no retrieved document context.

The input is truncated to a maximum of 512 tokens and generation is capped at 140 new tokens. Deterministic decoding (`do_sample=False`) makes repeated comparisons more consistent.

In [11]:
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 140
MAX_CONTEXT_CHARS = 5000


def generate_text(prompt, max_new_tokens=MAX_NEW_TOKENS):
    """Generate a bounded response with the loaded pretrained LLM."""
    if not LLM_READY or llm_model is None or llm_tokenizer is None:
        raise RuntimeError("FLAN-T5 is not loaded. Rerun the model-loading cell after checking internet access.")
    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )
    inputs = {key: value.to(LLM_DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=min(int(max_new_tokens), MAX_NEW_TOKENS),
            num_beams=2,
            do_sample=False,
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    return llm_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def build_context(retrieved_items, max_chars=MAX_CONTEXT_CHARS):
    """Format retrieved passages with source labels and cap prompt context size."""
    sections = []
    used_chars = 0
    for rank, item in enumerate(retrieved_items, start=1):
        label = f"[Source {rank}: {item['source']}, page {item['page']}]"
        section = f"{label}\n{item['text']}"
        remaining = max_chars - used_chars
        if remaining <= 0:
            break
        if len(section) > remaining:
            section = section[:remaining]
        sections.append(section)
        used_chars += len(section) + 2
        if used_chars >= max_chars:
            break
    return "\n\n".join(sections)


def answer_with_rag(question, top_k=DEFAULT_TOP_K):
    """Retrieve passages and generate an answer grounded in retrieved context."""
    retrieved = retrieve_documents(question, top_k=top_k)
    context = build_context(retrieved)
    prompt = f"""You are an academic question-answering assistant.
Answer the question using only the supplied context. Do not add facts that are not supported by the context.
If the context does not contain enough information, say: The documents do not provide enough information to answer this question.
Use clear, concise language.

Context:
{context}

Question: {question}

Answer:"""
    answer = generate_text(prompt)
    return answer, retrieved


def answer_without_retrieval(question):
    """Generate a baseline answer using the same LLM but no retrieved context."""
    prompt = f"""Answer the following academic question using your pretrained knowledge.
If you are uncertain, say so. Give a clear and concise answer.

Question: {question}

Answer:"""
    return generate_text(prompt)

print("Answer-generation functions are ready.")
print("Model ready:", LLM_READY)

Answer-generation functions are ready.
Model ready: True


## 11. Run a single RAG vs LLM-only comparison

Try a question that is covered by your uploaded notes. This cell prints both answers and the evidence retrieved for the RAG answer. Verify that the cited source passages actually support the generated response.

In [12]:
sample_question = "What is overfitting, and how can it be reduced?"

if LLM_READY:
    rag_answer, sample_retrieved = answer_with_rag(sample_question, top_k=4)
    llm_only_answer = answer_without_retrieval(sample_question)

    print("QUESTION:\n", sample_question)
    print("\nRAG ANSWER:\n", rag_answer)
    print("\nLLM-ONLY ANSWER:\n", llm_only_answer)
    print("\nRETRIEVED EVIDENCE:")
    display(pd.DataFrame(sample_retrieved)[["source", "page", "similarity", "text"]])
else:
    print("Skipped generation because FLAN-T5 did not load. Rerun the model-loading cell and then rerun this cell.")

QUESTION:
 What is overfitting, and how can it be reduced?

RAG ANSWER:
 A neural network is made of layers of connected computational units often called neurons. Each neuron computes a neural network.

LLM-ONLY ANSWER:
 weight loss

RETRIEVED EVIDENCE:


,source,page,similarity,text
0,Starter Notes - Model Evaluation and Overfitting,1,0.589383,when a model is too simple to capture importan...
1,Starter Notes - Model Evaluation and Overfitting,1,0.441218,A confusion matrix compares predicted classes ...
2,Starter Notes - Neural Networks and Deep Learning,1,0.439516,An artificial neural network is made of layers...
3,Starter Notes - Supervised Learning,1,0.359326,Supervised learning uses examples containing i...


## 12. Evaluate the system on a fixed question set

The following questions are drawn from the built-in AI/ML starter notes. If your own PDFs cover different topics, replace or add questions so that they test the contents of your documents. The cell runs a fixed number of questions; it does not loop indefinitely.

The generated comparison table records the actual outputs. Human scoring is intentionally left blank: do **not** invent evaluation scores. Read both answers and retrieved passages, then rate them using the rubric in the next section.

In [13]:
TEST_QUESTIONS = [
    "What is the difference between classification and regression?",
    "What is overfitting, and name methods that can reduce it.",
    "Define precision, recall, and F1-score.",
    "What is the purpose of embeddings in a RAG system?",
    "How does a RAG pipeline produce an answer?"
]

if LLM_READY:
    evaluation_rows = []
    for q_no, question in enumerate(TEST_QUESTIONS, start=1):
        started = time.time()
        rag_answer, retrieved = answer_with_rag(question, top_k=4)
        llm_only_answer = answer_without_retrieval(question)
        elapsed = round(time.time() - started, 2)
        sources = "; ".join(
            f"{item['source']} (p. {item['page']}, score {item['similarity']:.3f})"
            for item in retrieved
        )
        evaluation_rows.append({
            "Q.No.": q_no,
            "Question": question,
            "RAG Answer": rag_answer,
            "LLM-only Answer": llm_only_answer,
            "Retrieved Sources": sources,
            "Time (seconds)": elapsed
        })

    evaluation_df = pd.DataFrame(evaluation_rows)
    display(evaluation_df)
    evaluation_df.to_csv("rag_vs_llm_evaluation.csv", index=False)
    print("Saved actual answer comparison to rag_vs_llm_evaluation.csv")
else:
    evaluation_df = pd.DataFrame()
    print("Evaluation not run because FLAN-T5 is not loaded. Rerun the model-loading cell first.")

,Q.No.,Question,RAG Answer,LLM-only Answer,Retrieved Sources,Time (seconds)
0,1,What is the difference between classification ...,A machine-learning workflow commonly includes ...,classification is the process of determining i...,"Starter Notes - Supervised Learning (p. 1, sco...",2.62
1,2,"What is overfitting, and name methods that can...",Overfitting occurs when a model is too simple ...,"weight loss, exercise, diet",Starter Notes - Model Evaluation and Overfitti...,0.96
2,3,"Define precision, recall, and F1-score.",Accuracy measurement should always be paired w...,F1-score is a measure of how fast a car can race.,Starter Notes - Model Evaluation and Overfitti...,1.46
3,4,What is the purpose of embeddings in a RAG sys...,Retrieval-Augmented Generation (RAG) combines ...,to make the system more efficient.,"Starter Notes - Embeddings, Retrieval and RAG ...",4.71
4,5,How does a RAG pipeline produce an answer?,Retrieval-Augmented Generation (RAG) combines ...,a continuous stream of water,"Starter Notes - Embeddings, Retrieval and RAG ...",5.12


Saved actual answer comparison to rag_vs_llm_evaluation.csv


## 13. Manual evaluation rubric and score sheet

Use a **1–5 scale** after reading each output and checking the retrieved source text. Apply the same criteria to both approaches where applicable.

| Score | Interpretation |
|---:|---|
| 1 | Very poor / mostly incorrect |
| 2 | Weak / important information missing |
| 3 | Acceptable / partly correct |
| 4 | Good / mostly correct and useful |
| 5 | Excellent / accurate, complete, and well supported |

**Evaluation criteria**
- **Relevance:** Does the response answer the question asked?
- **Accuracy:** Are the claims factually correct when checked against reliable course material?
- **Grounding:** For RAG, are the claims supported by the retrieved passages? For LLM-only, record N/A or assess whether the answer is supported by your document collection.
- **Completeness:** Does the response cover the important parts without unnecessary detail?
- **Hallucination control:** Does it avoid inventing unsupported facts, definitions, or source details?

Do not claim that RAG is always better. Retrieval can return irrelevant passages, and an LLM can still misinterpret or add unsupported information. Discuss at least one observed strength and one limitation of each approach.

In [14]:
# Fill these ratings after reviewing the actual outputs in evaluation_df.
# Leave a score blank until you have inspected the corresponding answer.
score_sheet = pd.DataFrame({
    "Q.No.": list(range(1, len(TEST_QUESTIONS) + 1)),
    "Question": TEST_QUESTIONS,
    "RAG Relevance (1-5)": [""] * len(TEST_QUESTIONS),
    "RAG Accuracy (1-5)": [""] * len(TEST_QUESTIONS),
    "RAG Grounding (1-5)": [""] * len(TEST_QUESTIONS),
    "RAG Completeness (1-5)": [""] * len(TEST_QUESTIONS),
    "RAG Hallucination Control (1-5)": [""] * len(TEST_QUESTIONS),
    "LLM-only Relevance (1-5)": [""] * len(TEST_QUESTIONS),
    "LLM-only Accuracy (1-5)": [""] * len(TEST_QUESTIONS),
    "LLM-only Completeness (1-5)": [""] * len(TEST_QUESTIONS),
    "LLM-only Hallucination Control (1-5)": [""] * len(TEST_QUESTIONS)
})
display(score_sheet)
score_sheet.to_csv("manual_evaluation_score_sheet.csv", index=False)
print("Blank scoring sheet saved. Enter only scores you can justify from your inspection.")

,Q.No.,Question,RAG Relevance (1-5),RAG Accuracy (1-5),RAG Grounding (1-5),RAG Completeness (1-5),RAG Hallucination Control (1-5),LLM-only Relevance (1-5),LLM-only Accuracy (1-5),LLM-only Completeness (1-5),LLM-only Hallucination Control (1-5)
0,1,What is the difference between classification ...,,,,,,,,,
1,2,"What is overfitting, and name methods that can...",,,,,,,,,
2,3,"Define precision, recall, and F1-score.",,,,,,,,,
3,4,What is the purpose of embeddings in a RAG sys...,,,,,,,,,
4,5,How does a RAG pipeline produce an answer?,,,,,,,,,


Blank scoring sheet saved. Enter only scores you can justify from your inspection.


## 14. Interactive Gradio application

The interface accepts a question and displays the RAG answer, LLM-only answer, and retrieved source passages. It uses the already-loaded models and FAISS index; it does not retrain or rebuild them. `prevent_thread_lock=True` allows the notebook cell to return instead of waiting indefinitely for the web server.

In [15]:
def run_gradio_comparison(question, top_k):
    if not LLM_READY:
        message = "FLAN-T5 is not loaded. Rerun the model-loading cell first."
        return message, message, "No retrieval performed."
    if not isinstance(question, str) or not question.strip():
        return "Please enter a question.", "Please enter a question.", "No retrieval performed."
    try:
        rag_answer, retrieved = answer_with_rag(question.strip(), top_k=int(top_k))
        baseline_answer = answer_without_retrieval(question.strip())
        source_lines = []
        for rank, item in enumerate(retrieved, start=1):
            source_lines.append(
                f"{rank}. {item['source']} | page {item['page']} | cosine similarity {item['similarity']:.3f}\n"
                f"   {item['text'][:700]}"
            )
        return rag_answer, baseline_answer, "\n\n".join(source_lines) if source_lines else "No matching passages found."
    except Exception as exc:
        error = f"Unable to generate an answer: {type(exc).__name__}: {exc}"
        return error, error, "Check that the models and index have loaded correctly."

with gr.Blocks(title="AI & ML RAG Knowledge Assistant") as demo:
    gr.Markdown("# AI & ML Academic Knowledge Assistant\nAsk a question and compare a document-grounded RAG answer with the same LLM without retrieval.")
    question_box = gr.Textbox(
        label="Enter your question",
        placeholder="Example: What is overfitting and how can it be reduced?",
        lines=2
    )
    top_k_slider = gr.Slider(minimum=1, maximum=min(8, max(1, vector_index.ntotal)), value=min(4, max(1, vector_index.ntotal)), step=1, label="Number of retrieved chunks (top-k)")
    run_button = gr.Button("Generate and compare", variant="primary")
    rag_output = gr.Textbox(label="RAG answer (uses retrieved documents)", lines=6)
    baseline_output = gr.Textbox(label="LLM-only answer (no retrieval)", lines=6)
    source_output = gr.Textbox(label="Retrieved passages and source references", lines=10)
    run_button.click(
        fn=run_gradio_comparison,
        inputs=[question_box, top_k_slider],
        outputs=[rag_output, baseline_output, source_output]
    )

if LLM_READY:
    try:
        demo.launch(share=True, prevent_thread_lock=True, debug=False)
    except Exception as exc:
        print("Could not create a public Gradio share link:", type(exc).__name__, exc)
        print("Trying a local launch. In Colab, the public link may be unavailable due to network policy.")
        demo.launch(share=False, prevent_thread_lock=True, debug=False)
else:
    print("Gradio interface is defined but not launched because the LLM is not loaded.")
    print("Rerun the model-loading cell, then rerun this interface cell.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://300787786f9fd80d44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 15. Results and observations

After running the evaluation and the Gradio demo, complete the following observations using your actual outputs.

1. **Document processing:** State the number of PDFs, pages/sections, and chunks processed. Mention any scanned or unreadable files.
2. **Embedding model:** State the embedding model name and vector dimension printed in Section 6.
3. **Vector database:** State the FAISS index type and number of indexed vectors printed in Section 7.
4. **Retrieval behaviour:** Identify whether the top retrieved passages were relevant to each test question. Refer to their source filenames, page numbers, and similarity scores.
5. **Response comparison:** Compare the RAG and LLM-only answers in the saved evaluation table. Give at least two specific examples from your own run.
6. **Limitations:** Discuss retrieval misses, poor PDF text extraction, chunk-size sensitivity, context truncation, possible hallucinations, and dependence on pretrained model knowledge.

### Report-ready results table

| Measure | Actual result to enter |
|---|---|
| Number of PDFs uploaded | Enter observed count |
| Number of pages/sections | Enter observed count |
| Number of chunks | Enter printed count |
| Embedding model | `all-MiniLM-L6-v2` |
| Embedding dimension | Enter printed value (expected 384) |
| Vector database | FAISS `IndexFlatIP` |
| Number of indexed vectors | Enter printed count |
| Generator LLM | `google/flan-t5-base` |
| Retrieval top-k | 4 (default) |
| Evaluation questions | 5 (provided test set) |

### Limitations and future improvements
- Search quality depends on document quality, chunk size, overlap, and embedding-model suitability.
- Text extraction may fail for scanned PDFs, tables, diagrams, or complex layouts.
- Top-k retrieval may omit relevant information or include unrelated passages.
- The LLM may misinterpret retrieved context or generate unsupported details despite grounding instructions.
- The 512-token input limit can truncate long context; larger-context models or context reranking could improve this.
- Future work could add OCR, metadata filters, reranking, source-level citations, a larger domain-specific corpus, and human-reviewed question-answer benchmarks.


## 16. Conclusion (complete after observing your run)

**Conclusion template — edit the bracketed fields using your actual run:**

In this practical, a Retrieval-Augmented Generation application was developed for an AI and Machine Learning document collection. PDF text was extracted and cleaned, divided into overlapping chunks of **180 words** with **40-word overlap**, and encoded using the pretrained `all-MiniLM-L6-v2` sentence-transformer model. The resulting **[enter number]** vectors were stored in a FAISS `IndexFlatIP` vector index. For a user's question, the system retrieved the top four similar chunks and supplied them as context to the pretrained `google/flan-t5-base` language model. The same LLM was also evaluated without retrieval to create a baseline.

Across **[enter number]** test questions, the observed results showed that **[summarise what your comparison actually showed]**. The experiment demonstrated that retrieval can provide document-specific context and source traceability, while also showing that answer quality depends on the quality of the documents and retrieved passages. RAG reduces reliance on the LLM's general pretrained knowledge for the answer, but it does not eliminate retrieval errors or hallucinations.



















## 17. References

1. Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS. https://arxiv.org/abs/2005.11401
2. Reimers, N. and Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. https://arxiv.org/abs/1908.10084
3. Johnson, J., Douze, M., and Jégou, H. (2017). *Billion-scale similarity search with GPUs*. https://arxiv.org/abs/1702.08734
4. Raffel, C. et al. (2020). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*. https://jmlr.org/papers/v21/20-074.html
5. Hugging Face model card: `google/flan-t5-base`. https://huggingface.co/google/flan-t5-base
6. Sentence Transformers documentation. https://www.sbert.net/
7. FAISS documentation. https://github.com/facebookresearch/faiss
8. PyPDF documentation. https://pypdf.readthedocs.io/
9. Gradio documentation. https://www.gradio.app/docs






